# Phase 5: Targeted Analysis 11: Bootstrap Confidence Intervals

## Overview
Computes bootstrap 95% CIs (N=10,000, percentile method) for three key paper claims currently reported as point estimates.

## Claims
1. **Transfer efficiency** (81-96%): band-specific edges transfer generically across bands
2. **Majority-shared gap closure** (>=96%): edges shared by k>=3 bands recover full circuit accuracy (160m+)
3. **Cross-draw transfer ratio** (0.987-1.004): circuits from different draws are functionally equivalent

## Bootstrap Strategy
- **Paired resampling** for Claims 1 & 3 (ratio of correlated means)
- **Simple mean bootstrap** for Claim 2 (single array)
- Per-model CIs + aggregate CIs for "160m+" or "all models"

## Data Sources
- 'cross_band_eval_results.csv' (600 rows): Claim 1
- 'universal_core_eval_results.csv' (120 rows): Claim 1 baseline
- 'tiered_sharing_eval.csv' (300 rows): Claim 2
- 'cross_draw_eval.csv' (180 rows): Claim 3

## Notebook Structure
1. Setup & Imports
2. Load Data
3. Claim 1: Transfer Efficiency CIs
4. Claim 2: Gap Closure CIs
5. Claim 3: Cross-Draw Transfer Ratio CIs
6. Combine & Save (CSV + LaTeX)
7. Forest Plot
8. Bootstrap Distributions
9. Summary

## Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple

# Paths
PHASE5_DIR = Path("LSC_circuit_analysis/05_Phase_Targeted")
ANALYSIS_DIR = PHASE5_DIR / "outputs" / "analysis"
VIZ_DIR = PHASE5_DIR / "outputs" / "viz"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Constants
N_BOOTSTRAP = 10_000
RANDOM_SEED = 42
CI_LEVEL = 0.95
MODELS = ["pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
MODELS_160P = ["pythia-160m", "pythia-410m", "pythia-1b", "pythia-1.4b"]
BANDS = ["low", "medium", "high", "top", "very_top"]

# Style
sns.set_theme(style="whitegrid", font_scale=1.1)
MODEL_COLORS = {
    "pythia-70m": "#1f77b4",
    "pythia-160m": "#ff7f0e",
    "pythia-410m": "#2ca02c",
    "pythia-1b": "#d62728",
    "pythia-1.4b": "#9467bd",
    "160m+": "#9467bd",
    "all": "#8c564b",
}


def bootstrap_ci(
    data, statistic=np.mean, n_boot=N_BOOTSTRAP, ci=CI_LEVEL, seed=RANDOM_SEED
) -> Tuple[float, float, float]:
    """Bootstrap confidence interval. Returns (lower, upper, point_estimate)."""
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    if len(data) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.RandomState(seed)
    boot_stats = np.array(
        [
            statistic(rng.choice(data, size=len(data), replace=True))
            for _ in range(n_boot)
        ]
    )
    alpha = (1 - ci) / 2
    lower = float(np.percentile(boot_stats, 100 * alpha))
    upper = float(np.percentile(boot_stats, 100 * (1 - alpha)))
    point = float(statistic(data))
    return (lower, upper, point)


def bootstrap_ratio_ci(
    numer_vals, denom_vals, n_boot=N_BOOTSTRAP, ci=CI_LEVEL, seed=RANDOM_SEED
):
    """Bootstrap CI for ratio of means with paired resampling.

    numer_vals and denom_vals must be same length (paired).
    Returns (lower, upper, point_estimate, boot_distribution).
    """
    numer_vals = np.asarray(numer_vals, dtype=float)
    denom_vals = np.asarray(denom_vals, dtype=float)
    assert len(numer_vals) == len(denom_vals), (
        "Arrays must be same length for paired resampling"
    )
    n = len(numer_vals)
    rng = np.random.RandomState(seed)
    boot_ratios = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        boot_ratios[i] = numer_vals[idx].mean() / denom_vals[idx].mean()
    alpha = (1 - ci) / 2
    lower = float(np.percentile(boot_ratios, 100 * alpha))
    upper = float(np.percentile(boot_ratios, 100 * (1 - alpha)))
    point = float(numer_vals.mean() / denom_vals.mean())
    return (lower, upper, point, boot_ratios)


def bootstrap_grouped_ratio_ci(
    groups, n_boot=N_BOOTSTRAP, ci=CI_LEVEL, seed=RANDOM_SEED
):
    """Bootstrap CI for ratio of means with group-aware resampling.

    groups: list of dicts with 'same' (float) and 'cross' (list of floats).
    Returns (lower, upper, point_estimate, boot_distribution).
    """
    n = len(groups)
    rng = np.random.RandomState(seed)
    boot_ratios = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        same_vals = [groups[j]["same"] for j in idx]
        cross_vals = [v for j in idx for v in groups[j]["cross"]]
        boot_ratios[i] = np.mean(cross_vals) / np.mean(same_vals)
    # Point estimate
    all_same = [g["same"] for g in groups]
    all_cross = [v for g in groups for v in g["cross"]]
    point = float(np.mean(all_cross) / np.mean(all_same))
    alpha = (1 - ci) / 2
    lower = float(np.percentile(boot_ratios, 100 * alpha))
    upper = float(np.percentile(boot_ratios, 100 * (1 - alpha)))
    return (lower, upper, point, boot_ratios)


def save_figure(fig, name):
    path = VIZ_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


print("Setup complete.")

Setup complete.


## Load Data

In [2]:
df_cross_band = pd.read_csv(ANALYSIS_DIR / "cross_band_eval_results.csv")
df_universal = pd.read_csv(ANALYSIS_DIR / "universal_core_eval_results.csv")
df_tiered = pd.read_csv(ANALYSIS_DIR / "tiered_sharing_eval.csv")
df_cross_draw = pd.read_csv(ANALYSIS_DIR / "cross_draw_eval.csv")

print(f"cross_band_eval_results: {df_cross_band.shape}")
print(f"  columns: {list(df_cross_band.columns)}")
print(f"universal_core_eval_results: {df_universal.shape}")
print(f"  columns: {list(df_universal.columns)}")
print(f"tiered_sharing_eval: {df_tiered.shape}")
print(f"  columns: {list(df_tiered.columns)}")
print(f"cross_draw_eval: {df_cross_draw.shape}")
print(f"  columns: {list(df_cross_draw.columns)}")

# Filter cross_band to k_sample=0 (full band-specific circuit, no subsampling)
df_cb = df_cross_band[df_cross_band["k_sample"] == 0].copy()
print(f"\ncross_band after k_sample=0 filter: {df_cb.shape}")

# Filter universal to circuit_type='universal'
df_univ = df_universal[df_universal["circuit_type"] == "universal"].copy()
print(f"universal after circuit_type=universal filter: {df_univ.shape}")

cross_band_eval_results: (600, 9)
  columns: ['model', 'draw', 'source_band', 'test_band', 'circuit_type', 'k_sample', 'n_edges', 'accuracy', 'top5_accuracy']
universal_core_eval_results: (150, 8)
  columns: ['model', 'draw', 'test_band', 'circuit_type', 'n_edges', 'accuracy', 'top5_accuracy', 'mean_correct_prob']
tiered_sharing_eval: (375, 11)
  columns: ['model', 'draw', 'threshold_k', 'test_band', 'n_edges', 'accuracy', 'top5_accuracy', 'universal_acc', 'full_circuit_acc', 'gap', 'gap_closed']
cross_draw_eval: (225, 8)
  columns: ['model', 'band', 'source_draw', 'test_draw', 'is_same_draw', 'n_edges', 'accuracy', 'top5_accuracy']

cross_band after k_sample=0 filter: (300, 9)
universal after circuit_type=universal filter: (75, 8)


## Claim 1: Transfer Efficiency CIs

In [3]:
# For each (model, draw, test_band):
#   same_boost = accuracy(source_band==test_band) - universal_acc
#   cross_boost = mean accuracy(source_band!=test_band) - universal_acc
# Transfer efficiency = mean(cross_boost) / mean(same_boost)

# Merge universal accuracy into cross_band data
df_univ_acc = df_univ[["model", "draw", "test_band", "accuracy"]].rename(
    columns={"accuracy": "universal_acc"}
)
df_cb = df_cb.merge(df_univ_acc, on=["model", "draw", "test_band"], how="left")
df_cb["boost"] = df_cb["accuracy"] - df_cb["universal_acc"]

# Split into same-band and cross-band
df_cb["is_same_band"] = df_cb["source_band"] == df_cb["test_band"]

# Build paired arrays per model: for each (draw, test_band),
# same_boost is the boost when source==test, cross_boost is mean boost when source!=test
claim1_results = []
claim1_boot_dist = {}  # store boot distributions for aggregate

for model in MODELS:
    m_same = df_cb[(df_cb["model"] == model) & df_cb["is_same_band"]]
    m_cross = df_cb[(df_cb["model"] == model) & ~df_cb["is_same_band"]]

    # For each (draw, test_band), get same_boost and mean cross_boost
    same_boosts = []
    cross_boosts = []
    for (draw, tb), grp in m_cross.groupby(["draw", "test_band"]):
        same_row = m_same[(m_same["draw"] == draw) & (m_same["test_band"] == tb)]
        if len(same_row) == 0:
            continue
        same_boosts.append(same_row["boost"].values[0])
        cross_boosts.append(grp["boost"].mean())

    same_boosts = np.array(same_boosts)
    cross_boosts = np.array(cross_boosts)

    lower, upper, point, boot_dist = bootstrap_ratio_ci(cross_boosts, same_boosts)
    claim1_results.append(
        {
            "claim": "Transfer Efficiency",
            "model": model,
            "n_pairs": len(same_boosts),
            "point_estimate": point,
            "ci_lower": lower,
            "ci_upper": upper,
        }
    )
    print(
        f"{model}: TE = {point:.3f} [{lower:.3f}, {upper:.3f}] (n={len(same_boosts)} pairs)"
    )

# Aggregate: 160m+ (pool all pairs from 160m, 410m, 1b)
agg_same = []
agg_cross = []
for model in MODELS_160P:
    m_same = df_cb[(df_cb["model"] == model) & df_cb["is_same_band"]]
    m_cross = df_cb[(df_cb["model"] == model) & ~df_cb["is_same_band"]]
    for (draw, tb), grp in m_cross.groupby(["draw", "test_band"]):
        same_row = m_same[(m_same["draw"] == draw) & (m_same["test_band"] == tb)]
        if len(same_row) == 0:
            continue
        agg_same.append(same_row["boost"].values[0])
        agg_cross.append(grp["boost"].mean())

agg_same = np.array(agg_same)
agg_cross = np.array(agg_cross)
lower, upper, point, boot_dist_te = bootstrap_ratio_ci(agg_cross, agg_same)
claim1_results.append(
    {
        "claim": "Transfer Efficiency",
        "model": "160m+",
        "n_pairs": len(agg_same),
        "point_estimate": point,
        "ci_lower": lower,
        "ci_upper": upper,
    }
)
claim1_boot_dist["160m+"] = boot_dist_te
print(
    f"160m+ aggregate: TE = {point:.3f} [{lower:.3f}, {upper:.3f}] (n={len(agg_same)} pairs)"
)

df_claim1 = pd.DataFrame(claim1_results)
print(f"\nClaim 1 results: {len(df_claim1)} rows")
df_claim1

pythia-70m: TE = 0.814 [0.743, 0.914] (n=15 pairs)


pythia-160m: TE = 0.926 [0.892, 0.958] (n=15 pairs)


pythia-410m: TE = 0.964 [0.947, 0.984] (n=15 pairs)


pythia-1b: TE = 0.944 [0.918, 0.971] (n=15 pairs)


<TMPDIR>/ipykernel_503756/2877138726.py:69: RuntimeWarning: Mean of empty slice.
  boot_ratios[i] = numer_vals[idx].mean() / denom_vals[idx].mean()
<TMPDIR>/env/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


<TMPDIR>/ipykernel_503756/2877138726.py:73: RuntimeWarning: Mean of empty slice.
  point = float(numer_vals.mean() / denom_vals.mean())


pythia-1.4b: TE = nan [nan, nan] (n=0 pairs)


160m+ aggregate: TE = 0.947 [0.932, 0.963] (n=45 pairs)

Claim 1 results: 6 rows


,claim,model,n_pairs,point_estimate,ci_lower,ci_upper
0,Transfer Efficiency,pythia-70m,15,0.814320,0.742897,0.913908
1,Transfer Efficiency,pythia-160m,15,0.926170,0.892442,0.957817
2,Transfer Efficiency,pythia-410m,15,0.964450,0.946901,0.983577
3,Transfer Efficiency,pythia-1b,15,0.944284,0.917752,0.970707
4,Transfer Efficiency,pythia-1.4b,0,NaN,NaN,NaN
5,Transfer Efficiency,160m+,45,0.947101,0.931682,0.962524


## Claim 2: Gap Closure CIs

In [4]:
# Filter to threshold_k=3
df_k3 = df_tiered[df_tiered["threshold_k"] == 3].copy()
print(f"tiered_sharing at k=3: {df_k3.shape}")

claim2_results = []
claim2_boot_dist = {}

for model in MODELS:
    vals = df_k3[df_k3["model"] == model]["gap_closed"].values
    lower, upper, point = bootstrap_ci(vals)
    # Also compute min bootstrap for worst-case
    lower_min, upper_min, point_min = bootstrap_ci(vals, statistic=np.min)
    claim2_results.append(
        {
            "claim": "Gap Closure (k>=3)",
            "model": model,
            "n_values": len(vals),
            "point_estimate": point,
            "ci_lower": lower,
            "ci_upper": upper,
            "worst_case_point": point_min,
            "worst_case_lower": lower_min,
            "worst_case_upper": upper_min,
        }
    )
    print(
        f"{model}: gap_closed = {point:.4f} [{lower:.4f}, {upper:.4f}] "
        f"(min: {point_min:.4f} [{lower_min:.4f}, {upper_min:.4f}]) (n={len(vals)})"
    )

# Aggregate: 160m+
agg_vals = df_k3[df_k3["model"].isin(MODELS_160P)]["gap_closed"].values
lower, upper, point = bootstrap_ci(agg_vals)
lower_min, upper_min, point_min = bootstrap_ci(agg_vals, statistic=np.min)

# Store boot distribution for aggregate
rng = np.random.RandomState(RANDOM_SEED)
boot_gc = np.array(
    [
        np.mean(rng.choice(agg_vals, size=len(agg_vals), replace=True))
        for _ in range(N_BOOTSTRAP)
    ]
)
claim2_boot_dist["160m+"] = boot_gc

claim2_results.append(
    {
        "claim": "Gap Closure (k>=3)",
        "model": "160m+",
        "n_values": len(agg_vals),
        "point_estimate": point,
        "ci_lower": lower,
        "ci_upper": upper,
        "worst_case_point": point_min,
        "worst_case_lower": lower_min,
        "worst_case_upper": upper_min,
    }
)
print(
    f"160m+ aggregate: gap_closed = {point:.4f} [{lower:.4f}, {upper:.4f}] "
    f"(min: {point_min:.4f} [{lower_min:.4f}, {upper_min:.4f}]) (n={len(agg_vals)})"
)

df_claim2 = pd.DataFrame(claim2_results)
print(f"\nClaim 2 results: {len(df_claim2)} rows")
df_claim2

tiered_sharing at k=3: (75, 11)


pythia-70m: gap_closed = 0.9931 [0.7131, 1.4027] (min: 0.3559 [0.3559, 0.6797]) (n=15)


pythia-160m: gap_closed = 1.0092 [0.9796, 1.0449] (min: 0.9251 [0.9251, 0.9655]) (n=15)


pythia-410m: gap_closed = 1.0092 [0.9953, 1.0225] (min: 0.9412 [0.9412, 0.9923]) (n=15)


pythia-1b: gap_closed = 1.0526 [1.0401, 1.0670] (min: 1.0190 [1.0190, 1.0350]) (n=15)


pythia-1.4b: gap_closed = 1.0549 [1.0325, 1.0762] (min: 0.9561 [0.9561, 1.0195]) (n=15)


160m+ aggregate: gap_closed = 1.0315 [1.0193, 1.0439] (min: 0.9251 [0.9251, 0.9534]) (n=60)

Claim 2 results: 6 rows


,claim,model,n_values,point_estimate,ci_lower,ci_upper,worst_case_point,worst_case_lower,worst_case_upper
0,Gap Closure (k>=3),pythia-70m,15,0.993149,0.713052,1.402737,0.355932,0.355932,0.679687
1,Gap Closure (k>=3),pythia-160m,15,1.009156,0.979612,1.044941,0.925150,0.925150,0.965517
2,Gap Closure (k>=3),pythia-410m,15,1.009236,0.995324,1.022488,0.941176,0.941176,0.992308
3,Gap Closure (k>=3),pythia-1b,15,1.052647,1.040060,1.067028,1.019022,1.019022,1.035040
4,Gap Closure (k>=3),pythia-1.4b,15,1.054925,1.032535,1.076190,0.956107,0.956107,1.019531
5,Gap Closure (k>=3),160m+,60,1.031491,1.019319,1.043903,0.925150,0.925150,0.953390


## Claim 3: Cross-Draw Transfer Ratio CIs

In [5]:
# Group by (band, source_draw) per model.
# Each group has 1 same-draw acc (is_same_draw=True) and 2 cross-draw accs.
# Ratio = mean(cross_draw_acc) / mean(same_draw_acc)

claim3_results = []
claim3_boot_dist = {}

for model in MODELS:
    m_data = df_cross_draw[df_cross_draw["model"] == model]
    groups = []
    for (band, src_draw), grp in m_data.groupby(["band", "source_draw"]):
        same = grp[grp["is_same_draw"] == True]["accuracy"].values
        cross = grp[grp["is_same_draw"] == False]["accuracy"].values
        if len(same) == 0:
            continue
        groups.append({"same": same[0], "cross": list(cross)})

    lower, upper, point, boot_dist = bootstrap_grouped_ratio_ci(groups)
    claim3_results.append(
        {
            "claim": "Cross-Draw Transfer Ratio",
            "model": model,
            "n_groups": len(groups),
            "point_estimate": point,
            "ci_lower": lower,
            "ci_upper": upper,
        }
    )
    print(
        f"{model}: ratio = {point:.4f} [{lower:.4f}, {upper:.4f}] (n={len(groups)} groups)"
    )

# Aggregate: all models
all_groups = []
for model in MODELS:
    m_data = df_cross_draw[df_cross_draw["model"] == model]
    for (band, src_draw), grp in m_data.groupby(["band", "source_draw"]):
        same = grp[grp["is_same_draw"] == True]["accuracy"].values
        cross = grp[grp["is_same_draw"] == False]["accuracy"].values
        if len(same) == 0:
            continue
        all_groups.append({"same": same[0], "cross": list(cross)})

lower, upper, point, boot_dist_cd = bootstrap_grouped_ratio_ci(all_groups)
claim3_results.append(
    {
        "claim": "Cross-Draw Transfer Ratio",
        "model": "all",
        "n_groups": len(all_groups),
        "point_estimate": point,
        "ci_lower": lower,
        "ci_upper": upper,
    }
)
claim3_boot_dist["all"] = boot_dist_cd
print(
    f"All models aggregate: ratio = {point:.4f} [{lower:.4f}, {upper:.4f}] (n={len(all_groups)} groups)"
)

df_claim3 = pd.DataFrame(claim3_results)
print(f"\nClaim 3 results: {len(df_claim3)} rows")
df_claim3

pythia-70m: ratio = 0.9904 [0.9354, 1.0467] (n=15 groups)


pythia-160m: ratio = 0.9982 [0.9882, 1.0084] (n=15 groups)


pythia-410m: ratio = 0.9989 [0.9925, 1.0060] (n=15 groups)


pythia-1b: ratio = 1.0035 [0.9932, 1.0143] (n=15 groups)


pythia-1.4b: ratio = 1.0169 [1.0027, 1.0310] (n=15 groups)


All models aggregate: ratio = 1.0028 [0.9950, 1.0103] (n=75 groups)

Claim 3 results: 6 rows


,claim,model,n_groups,point_estimate,ci_lower,ci_upper
0,Cross-Draw Transfer Ratio,pythia-70m,15,0.990398,0.935431,1.046696
1,Cross-Draw Transfer Ratio,pythia-160m,15,0.998233,0.988175,1.008444
2,Cross-Draw Transfer Ratio,pythia-410m,15,0.998924,0.992514,1.006041
3,Cross-Draw Transfer Ratio,pythia-1b,15,1.003502,0.993220,1.014254
4,Cross-Draw Transfer Ratio,pythia-1.4b,15,1.016852,1.002683,1.030996
5,Cross-Draw Transfer Ratio,all,75,1.002774,0.994985,1.010278


## Combine & Save (CSV + LaTeX)

In [6]:
# Standardize columns across claims
def standardize_claim_df(df, n_col):
    return df[
        ["claim", "model", n_col, "point_estimate", "ci_lower", "ci_upper"]
    ].rename(columns={n_col: "n_samples"})


df_all_ci = pd.concat(
    [
        standardize_claim_df(df_claim1, "n_pairs"),
        standardize_claim_df(df_claim2, "n_values"),
        standardize_claim_df(df_claim3, "n_groups"),
    ],
    ignore_index=True,
)

# Add formatted CI string and width
df_all_ci["ci_width"] = df_all_ci["ci_upper"] - df_all_ci["ci_lower"]
df_all_ci["ci_string"] = df_all_ci.apply(
    lambda r: f"{r['point_estimate']:.3f} [{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]",
    axis=1,
)

# Save CSV
csv_path = ANALYSIS_DIR / "bootstrap_ci_summary.csv"
df_all_ci.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(f"Total rows: {len(df_all_ci)}")

# Generate LaTeX table
latex_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Bootstrap 95\% Confidence Intervals for Key Claims (N=10{,}000)}",
    r"\label{tab:bootstrap_ci}",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Claim & Model & Point Est. & 95\% CI & CI Width \\",
    r"\midrule",
]

prev_claim = None
for _, row in df_all_ci.iterrows():
    claim_label = row["claim"] if row["claim"] != prev_claim else ""
    if row["claim"] != prev_claim and prev_claim is not None:
        latex_lines.append(r"\midrule")
    prev_claim = row["claim"]

    model_label = row["model"].replace("pythia-", "").replace("160m+", r"$\geq$160m")
    pe = f"{row['point_estimate']:.3f}"
    ci = f"[{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]"
    width = f"{row['ci_width']:.3f}"
    latex_lines.append(f"{claim_label} & {model_label} & {pe} & {ci} & {width} \\\\")

latex_lines.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
)

tex_path = ANALYSIS_DIR / "bootstrap_ci_table.tex"
tex_path.write_text("\n".join(latex_lines))
print(f"Saved: {tex_path}")

df_all_ci

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/bootstrap_ci_summary.csv
Total rows: 18
Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/analysis/bootstrap_ci_table.tex


,claim,model,n_samples,point_estimate,ci_lower,ci_upper,ci_width,ci_string
0,Transfer Efficiency,pythia-70m,15,0.814320,0.742897,0.913908,0.171011,"0.814 [0.743, 0.914]"
1,Transfer Efficiency,pythia-160m,15,0.926170,0.892442,0.957817,0.065375,"0.926 [0.892, 0.958]"
2,Transfer Efficiency,pythia-410m,15,0.964450,0.946901,0.983577,0.036676,"0.964 [0.947, 0.984]"
3,Transfer Efficiency,pythia-1b,15,0.944284,0.917752,0.970707,0.052955,"0.944 [0.918, 0.971]"
4,Transfer Efficiency,pythia-1.4b,0,NaN,NaN,NaN,NaN,"nan [nan, nan]"
5,Transfer Efficiency,160m+,45,0.947101,0.931682,0.962524,0.030842,"0.947 [0.932, 0.963]"
6,Gap Closure (k>=3),pythia-70m,15,0.993149,0.713052,1.402737,0.689685,"0.993 [0.713, 1.403]"
7,Gap Closure (k>=3),pythia-160m,15,1.009156,0.979612,1.044941,0.065329,"1.009 [0.980, 1.045]"
8,Gap Closure (k>=3),pythia-410m,15,1.009236,0.995324,1.022488,0.027164,"1.009 [0.995, 1.022]"
9,Gap Closure (k>=3),pythia-1b,15,1.052647,1.040060,1.067028,0.026968,"1.053 [1.040, 1.067]"


## Forest Plot

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

claim_configs = [
    ("Transfer Efficiency", "Transfer Efficiency (ratio)", None),
    ("Gap Closure (k>=3)", "Gap Closure at k>=3 (fraction)", 1.0),
    ("Cross-Draw Transfer Ratio", "Cross-Draw Transfer Ratio", 1.0),
]

for ax, (claim_name, title, ref_line) in zip(axes, claim_configs):
    subset = df_all_ci[df_all_ci["claim"] == claim_name].copy()
    subset = subset.iloc[::-1]  # reverse so first model is at top

    y_pos = range(len(subset))
    colors = [MODEL_COLORS.get(m, "#333333") for m in subset["model"]]

    for i, (_, row) in enumerate(subset.iterrows()):
        color = MODEL_COLORS.get(row["model"], "#333333")
        ax.errorbar(
            row["point_estimate"],
            i,
            xerr=[
                [row["point_estimate"] - row["ci_lower"]],
                [row["ci_upper"] - row["point_estimate"]],
            ],
            fmt="o",
            color=color,
            capsize=4,
            capthick=1.5,
            markersize=8,
            linewidth=1.5,
            label=row["model"].replace("pythia-", ""),
        )

    if ref_line is not None:
        ax.axvline(ref_line, color="gray", linestyle="--", alpha=0.5, linewidth=1)

    labels = [m.replace("pythia-", "") for m in subset["model"]]
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Value")

fig.suptitle(
    "Bootstrap 95% Confidence Intervals for Key Paper Claims",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
save_figure(fig, "T11_01_bootstrap_forest_plot.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T11_01_bootstrap_forest_plot.png


## Bootstrap Distributions

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dist_configs = [
    (
        claim1_boot_dist.get("160m+", np.array([])),
        "Transfer Efficiency (160m+)",
        df_all_ci[
            (df_all_ci["claim"] == "Transfer Efficiency")
            & (df_all_ci["model"] == "160m+")
        ],
    ),
    (
        claim2_boot_dist.get("160m+", np.array([])),
        "Gap Closure k>=3 (160m+)",
        df_all_ci[
            (df_all_ci["claim"] == "Gap Closure (k>=3)")
            & (df_all_ci["model"] == "160m+")
        ],
    ),
    (
        claim3_boot_dist.get("all", np.array([])),
        "Cross-Draw Ratio (all models)",
        df_all_ci[
            (df_all_ci["claim"] == "Cross-Draw Transfer Ratio")
            & (df_all_ci["model"] == "all")
        ],
    ),
]

for ax, (boot_dist, title, ci_row) in zip(axes, dist_configs):
    if len(boot_dist) == 0:
        ax.set_title(title + " (no data)")
        continue

    ax.hist(
        boot_dist,
        bins=80,
        color="steelblue",
        alpha=0.7,
        edgecolor="white",
        linewidth=0.3,
    )

    if len(ci_row) > 0:
        row = ci_row.iloc[0]
        ax.axvline(
            row["point_estimate"], color="red", linewidth=2, label="Point estimate"
        )
        ax.axvline(
            row["ci_lower"],
            color="orange",
            linewidth=1.5,
            linestyle="--",
            label="95% CI",
        )
        ax.axvline(row["ci_upper"], color="orange", linewidth=1.5, linestyle="--")
        ax.axvspan(row["ci_lower"], row["ci_upper"], alpha=0.1, color="orange")

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Bootstrap statistic")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

fig.suptitle(
    "Bootstrap Distributions (N=10,000)", fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
save_figure(fig, "T11_02_bootstrap_distributions.png")

Saved: LSC_circuit_analysis/05_Phase_Targeted/outputs/viz/T11_02_bootstrap_distributions.png


## Summary

In [9]:
print("=" * 70)
print("BOOTSTRAP CONFIDENCE INTERVALS: SUMMARY")
print("=" * 70)

for claim_name in df_all_ci["claim"].unique():
    print(f"\n--- {claim_name} ---")
    subset = df_all_ci[df_all_ci["claim"] == claim_name]
    for _, row in subset.iterrows():
        model_label = row["model"].replace("pythia-", "")
        print(
            f"  {model_label:>8s}: {row['ci_string']}  (width={row['ci_width']:.4f}, n={int(row['n_samples'])})"
        )

print(f"\n{'=' * 70}")
print("Output files:")
print(f"  CSV:   {ANALYSIS_DIR / 'bootstrap_ci_summary.csv'}")
print(f"  LaTeX: {ANALYSIS_DIR / 'bootstrap_ci_table.tex'}")
print(f"  Plot:  {VIZ_DIR / 'T11_01_bootstrap_forest_plot.png'}")
print(f"  Plot:  {VIZ_DIR / 'T11_02_bootstrap_distributions.png'}")
print(
    f"\nAll CIs computed with N_BOOTSTRAP={N_BOOTSTRAP}, CI_LEVEL={CI_LEVEL}, SEED={RANDOM_SEED}"
)

BOOTSTRAP CONFIDENCE INTERVALS: SUMMARY

--- Transfer Efficiency ---
       70m: 0.814 [0.743, 0.914]  (width=0.1710, n=15)
      160m: 0.926 [0.892, 0.958]  (width=0.0654, n=15)
      410m: 0.964 [0.947, 0.984]  (width=0.0367, n=15)
        1b: 0.944 [0.918, 0.971]  (width=0.0530, n=15)
      1.4b: nan [nan, nan]  (width=nan, n=0)
     160m+: 0.947 [0.932, 0.963]  (width=0.0308, n=45)

--- Gap Closure (k>=3) ---
       70m: 0.993 [0.713, 1.403]  (width=0.6897, n=15)
      160m: 1.009 [0.980, 1.045]  (width=0.0653, n=15)
      410m: 1.009 [0.995, 1.022]  (width=0.0272, n=15)
        1b: 1.053 [1.040, 1.067]  (width=0.0270, n=15)
      1.4b: 1.055 [1.033, 1.076]  (width=0.0437, n=15)
     160m+: 1.031 [1.019, 1.044]  (width=0.0246, n=60)

--- Cross-Draw Transfer Ratio ---
       70m: 0.990 [0.935, 1.047]  (width=0.1113, n=15)
      160m: 0.998 [0.988, 1.008]  (width=0.0203, n=15)
      410m: 0.999 [0.993, 1.006]  (width=0.0135, n=15)
        1b: 1.004 [0.993, 1.014]  (width=0.0210, n=15